# 🔵 Notebook 05 — K-Means: Segmentação de Padrões de Incidentes
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Identificar grupos naturais de incidentes com comportamentos similares,
revelando padrões operacionais que o modelo supervisionado não captura.

**Input:** `data/processed/incidents_features.parquet`
**Output:** `outputs/clusters.json`

**K escolhido:** 4 clusters (baseado na EDA — 4 padrões principais identificados)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
import json, os
from datetime import date

print("✅ Imports ok")


In [ ]:
# ── Carregar dados ────────────────────────────────────────────────────────────
df = pd.read_parquet('../data/processed/incidents_features.parquet')
print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")


In [ ]:
# ── Features para clustering ──────────────────────────────────────────────────
# Usar apenas features de comportamento — não usar target_ola (unsupervised)
FEATURES_CLUSTER = [
    'hora', 'dia_semana', 'is_horario_comercial', 'is_fim_de_semana',
    'periodo_dia', 'prioridade_bin',
    'rolling_7d', 'rolling_30d',
    'Produto_freq', 'Grupo designado_freq',
]

X = df[FEATURES_CLUSTER].copy()

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features usadas: {FEATURES_CLUSTER}")
print(f"Shape após scaling: {X_scaled.shape}")


In [ ]:
# ── Elbow Method — escolha de K ───────────────────────────────────────────────
inertias    = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_, sample_size=5000))
    print(f"K={k}: inertia={km.inertia_:.0f} | silhouette={silhouettes[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method')
ax2.plot(K_range, silhouettes, 'ro-')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette'); ax2.set_title('Silhouette Score')
plt.tight_layout()
plt.savefig('../outputs/kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Treinar modelo final com K=4 ─────────────────────────────────────────────
K_FINAL = 4
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
labels = km_final.fit_predict(X_scaled)

df['cluster'] = labels

sil  = silhouette_score(X_scaled, labels, sample_size=5000)
db   = davies_bouldin_score(X_scaled, labels)

print(f"K={K_FINAL}")
print(f"Silhouette Score: {sil:.4f} (maior = melhor, max=1)")
print(f"Davies-Bouldin:   {db:.4f} (menor = melhor)")
print(f"\nDistribuição por cluster:")
print(df['cluster'].value_counts().sort_index())


In [ ]:
# ── Análise dos clusters ──────────────────────────────────────────────────────
dias_nome  = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
periodos   = {0:'Madrugada', 1:'Manhã', 2:'Tarde', 3:'Noite'}

for c in range(K_FINAL):
    sub = df[df['cluster']==c]
    print(f"\n{'='*50}")
    print(f"CLUSTER {c} — {len(sub)} incidentes ({len(sub)/len(df)*100:.1f}%)")
    print(f"  Hora média:        {sub['hora'].mean():.1f}h")
    print(f"  Taxa violação OLA: {sub['target_ola'].mean()*100:.2f}%")
    print(f"  % fim de semana:   {sub['is_fim_de_semana'].mean()*100:.1f}%")
    print(f"  % horário comerc.: {sub['is_horario_comercial'].mean()*100:.1f}%")
    print(f"  Período dominante: {periodos.get(int(sub['periodo_dia'].mode()[0]), '?')}")
    dow_counts = sub['dia_semana'].value_counts().head(2)
    print(f"  Dias críticos:     {[dias_nome[d] for d in dow_counts.index.tolist()]}")


In [ ]:
# ── Visualização PCA 2D ───────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled[:10000])  # sample para velocidade
labels_sample = labels[:10000]

cores = ['#E8002D', '#ff9500', '#34c759', '#5ac8fa']
fig, ax = plt.subplots(figsize=(10, 7))
for c in range(K_FINAL):
    mask = labels_sample == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=cores[c], alpha=0.4, s=8, label=f'Cluster {c}')
ax.set_title('K-Means — Visualização PCA (amostra 10k)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/kmeans_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Variância explicada pelos 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%")


In [ ]:
# ── Exportar clusters.json ────────────────────────────────────────────────────
# Labels descritivos baseados na análise acima — ajustar após rodar
LABELS = [
    "Incidentes noturnos recorrentes",
    "Picos P2 horário comercial",
    "Volume P3 alto — OLA ok",
    "Anomalias grupo crítico",
]

clusters_out = []
dias_nome = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']

for c in range(K_FINAL):
    sub = df[df['cluster']==c]
    dow_top = sub['dia_semana'].value_counts().head(2).index.tolist()
    prod_top = sub['Produto_freq'].nlargest(2).index.tolist()  # freq = proxy do produto

    clusters_out.append({
        "id": c,
        "label": LABELS[c],
        "tamanho": int(len(sub)),
        "taxaViolacao": round(sub['target_ola'].mean() * 100, 2),
        "perfil": {
            "horaMedia": round(sub['hora'].mean(), 1),
            "diasCriticos": [dias_nome[d] for d in dow_top],
            "pctFimSemana": round(sub['is_fim_de_semana'].mean() * 100, 1),
            "pctHorarioComercial": round(sub['is_horario_comercial'].mean() * 100, 1),
        },
    })

output = {
    "modelo": "kmeans_incident_segmentation",
    "gerado_em": date.today().strftime('%Y-%m-%d'),
    "n_clusters": K_FINAL,
    "metricas": {
        "silhouette": round(sil, 4),
        "davies_bouldin": round(db, 4),
        "features_usadas": FEATURES_CLUSTER,
    },
    "clusters": clusters_out,
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/clusters.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ clusters.json exportado")
print(json.dumps(output, ensure_ascii=False, indent=2))
